# Chapter 6: Compact Surfaces

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 6, printed pages 159-182, PDF pages 177-200.

**Chapter Goal.** Build a computational picture of the compact connected surfaces that appear in Chapter 6: how they are assembled from edge-paired polygons, how connected sum changes the presentation word, how orientability and Euler characteristic are read from a schema, and why the classification theorem reduces every compact connected surface to a sphere, a connected sum of tori, or a connected sum of projective planes.

The chapter is about a useful translation: a compact surface can be studied through a finite amount of combinatorics. A polygonal region is a disk with named boundary edges. Pair every edge with exactly one mate, identify paired edges by affine maps, and the quotient is locally Euclidean even at the vertices because the incident wedges can be fanned into a disk. The topological surface is continuous and geometric; the notebook model is discrete. The visual work below keeps both views visible: the polygon tells us what is glued, the union-find computation tells us which vertices become one point, and the invariant table tells us which standard surface the presentation reduces to.

The source chapter also emphasizes a limitation that matters pedagogically. The Part I classification theorem shows that every compact connected surface is homeomorphic to a member of a standard list. At this point in the book, the list has not yet been proved to have no duplicates. Euler characteristic and orientability are therefore used here as presentation-level invariants and as a preview of later topological invariants. The code follows that same boundary: it classifies presentations by the standard algorithmic data, while the markdown notes where Chapter 10 will supply the missing separation proof.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-06-compact-surfaces/06-compact-surfaces.ipynb",
  "course_dir": "Introduction-to-Topological-Manifolds",
  "course_title": "Introduction to Topological Manifolds",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/chapter-06-compact-surfaces/06-compact-surfaces.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Introduction-to-Topological-Manifolds/chapter-06-compact-surfaces/06-compact-surfaces.ipynb",
  "notebook_title": "Chapter 6: Compact Surfaces",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/topology.txt",
  "runtime_profile": "topology"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


## Computational Translation Guide

| Topological object or move | Computational representation in this notebook | What to inspect |
| --- | --- | --- |
| Polygonal presentation | A cyclic word such as `a b a^-1 b^-1`, plus a union-find structure on boundary vertices | Which edge labels are paired and which vertices become one quotient point |
| Sphere, torus, projective plane, Klein bottle | Standard presentation words | Euler characteristic, edge-pair type, and orientability flag |
| Connected sum | Concatenation of one-face presentation words after relabeling | Euler characteristic is additive with the correction `chi(M # N) = chi(M) + chi(N) - 2` |
| Elementary transformations | Symbolic moves that preserve the geometric realization | The ledger checks that each move has zero net change in Euler characteristic |
| Classification theorem proof | A dependency graph and a normalized standard-surface table | How one-face presentations are simplified into sphere, torus sums, or projective-plane sums |
| Orientability | Every paired label appears once in each direction | Complementary pairs permit a consistent front/back choice; twisted pairs obstruct it |

The library routing is deliberately mixed. `matplotlib` is used for durable 2D schemas because edge arrows and labels need exact placement. `networkx` is used for proof dependency structure because the theorem is an algorithm of reductions. `plotly` is used for a hoverable classification atlas, where the learner can compare standard families without rerunning code. `sympy` is used only for exact formula checks; it is not a replacement for the topology, but it is useful for keeping the genus and Euler-characteristic algebra honest. The course-local `utils` package supplies artifact saving, display, and small topology helpers.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import sympy as sp
from IPython.display import display

start = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [start, *start.parents]:
    direct = candidate
    nested = candidate / "Introduction-to-Topological-Manifolds"
    if (direct / "source_map.json").exists() and (direct / "utils").exists():
        BOOK_ROOT = direct
        break
    if (nested / "source_map.json").exists() and (nested / "utils").exists():
        BOOK_ROOT = nested
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate Introduction-to-Topological-Manifolds root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (
    assert_artifacts,
    chapter_artifact_root,
    display_artifact,
    save_csv,
    save_json,
    save_matplotlib,
    save_plotly_html,
)
from utils.topology import euler_characteristic, orientability_from_schema
from utils.validation import image_stats, relative

UNIT_KEY = "chapter-06-compact-surfaces"
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
CHECKS = ARTIFACT_ROOT / "checks"
TABLES = ARTIFACT_ROOT / "tables"

plt.rcParams.update({
    "figure.dpi": 130,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
})

print(f"Book root: {BOOK_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT}")


## Library Routing And Visual Storyboard

The visual sequence is organized around inspection targets rather than renderer quotas. First, polygon schemas make edge identifications visible. Second, a classifier turns those schemas into cells, Euler characteristic, and orientability data. Third, a connected-sum atlas shows how the standard presentations accumulate handles or crosscaps. Fourth, a proof dependency map records the logic of the classification theorem. Finally, a lab lets the reader test new words and see which part of the standard list the word indicates.

The storyboard saved below is the notebook's implementation brief. It is intentionally stored as a JSON check artifact so a later quality-control pass can compare the promised visuals with the generated files.


In [ ]:
storyboard = [
    {
        "concept": "edge-paired quotient models",
        "representation": "four labeled polygon schemas with arrows",
        "library": "matplotlib",
        "artifact": "figures/polygon-schema-models.png",
        "inspection_target": "paired labels, direction of gluing, and quotient vertex classes",
        "validation": "computed V, E, F, Euler characteristic, and orientability for each schema",
    },
    {
        "concept": "connected sums and standard families",
        "representation": "classification atlas with hoverable standard surfaces",
        "library": "plotly",
        "artifact": "html/euler-orientability-classification-atlas.html",
        "inspection_target": "how genus or crosscap number controls Euler characteristic",
        "validation": "symbolic formulas chi=2-2g and chi=2-k plus connected-sum correction",
    },
    {
        "concept": "classification theorem proof scaffold",
        "representation": "directed dependency graph",
        "library": "networkx plus matplotlib",
        "artifact": "figures/classification-proof-dependency-map.png",
        "inspection_target": "which reductions lead from a triangulated compact surface to the standard list",
        "validation": "acyclic directed graph with expected theorem endpoint",
    },
    {
        "concept": "elementary transformations preserve Euler characteristic",
        "representation": "delta ledger table",
        "library": "pandas plus sympy arithmetic",
        "artifact": "tables/elementary-transformation-euler-ledger.csv",
        "inspection_target": "changes in cell counts cancel for each elementary move",
        "validation": "every recorded delta has dV-dE+dF=0",
    },
    {
        "concept": "applied classification lab",
        "representation": "data table for sample presentation words",
        "library": "pandas plus course-local topology helpers",
        "artifact": "tables/applied-lab-classification-results.csv",
        "inspection_target": "classification signal from Euler characteristic and edge-pair type",
        "validation": "standard examples match sphere, torus, projective plane, and Klein bottle expectations",
    },
]

routing_rows = [
    {"concept": "polygon schemas", "representation": "2D edge arrows", "library": "matplotlib", "why": "precise static labels and arrows are more legible than a 3D embedding"},
    {"concept": "quotient vertex classes", "representation": "union-find on boundary vertices", "library": "numpy plus Python", "why": "the invariant is finite combinatorics, not numerical geometry"},
    {"concept": "classification proof", "representation": "dependency graph", "library": "networkx", "why": "the theorem is a sequence of reductions with named lemmas"},
    {"concept": "standard family comparison", "representation": "hoverable atlas", "library": "plotly", "why": "interactive hover exposes genus, word, Euler characteristic, and orientability together"},
    {"concept": "Euler formulas", "representation": "exact symbolic checks", "library": "sympy", "why": "the formulas are small exact identities"},
    {"concept": "artifact display", "representation": "inline notebook calls", "library": "course utils.artifacts", "why": "keeps paths book-local and consistent with the course contract"},
]

storyboard_payload = {
    "chapter": "Chapter 6: Compact Surfaces",
    "source_span": "printed pages 159-182; PDF pages 177-200",
    "source_reading_note": "Inspected with pdftotext; PDF page 176 was also checked to capture the printed page 159 opener because the provided PDF span begins at printed page 160 in this local file.",
    "visual_sequence": storyboard,
    "library_routing": routing_rows,
}
visual_storyboard_path = save_json(storyboard_payload, CHECKS / "visual-storyboard.json")
library_routing_path = save_csv(routing_rows, TABLES / "library-routing.csv")

display(pd.DataFrame(routing_rows))
display_artifact(visual_storyboard_path)


## Polygon Schemas As Finite Surface Data

A one-face surface presentation is a cyclic boundary word. Each letter appears exactly twice, counting either direction as an occurrence. The polygon has one 2-cell. Each label pair becomes one 1-cell. The only subtle count is the number of 0-cells, because several polygon vertices may be identified after the edge pairings are imposed. The union-find computation below follows the definition: for a positive edge label, the initial endpoint is the counterclockwise start of the edge; for an inverse label, the initial endpoint is the counterclockwise end. Paired edges identify initial endpoints with initial endpoints and terminal endpoints with terminal endpoints.

This finite model is not a proof that every compact surface has such a presentation. That result comes from triangulating a compact surface and observing that every 1-simplex lies in two 2-simplices. The finite model is what the proof produces after the triangulation theorem has done the geometric work. Once the schema is available, the essential chapter data are visible: edge-pair type, orientability, Euler characteristic, and the standard family suggested by the classification theorem.


In [ ]:
def parse_token(token):
    token = str(token).strip()
    if token.startswith("-"):
        return token[1:], -1
    if token.endswith("^-1"):
        return token[:-3], -1
    return token, 1


def render_token(token):
    label, sign = parse_token(token)
    return f"{label}^-1" if sign < 0 else label


def word_text(word):
    return " ".join(render_token(token) for token in word)


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra

    def classes(self):
        buckets = {}
        for item in range(len(self.parent)):
            buckets.setdefault(self.find(item), []).append(item)
        return [tuple(values) for values in sorted(buckets.values(), key=lambda v: v[0])]


def vertex_classes_for_word(word):
    n = len(word)
    uf = UnionFind(n)
    occurrences = {}
    for i, token in enumerate(word):
        label, sign = parse_token(token)
        ccw_start = i
        ccw_end = (i + 1) % n
        initial, terminal = (ccw_start, ccw_end) if sign > 0 else (ccw_end, ccw_start)
        occurrences.setdefault(label, []).append({
            "edge_index": i,
            "sign": sign,
            "initial": initial,
            "terminal": terminal,
        })
    bad = {label: occ for label, occ in occurrences.items() if len(occ) != 2}
    if bad:
        raise ValueError(f"Surface presentation needs each label exactly twice: {bad}")
    for occ in occurrences.values():
        uf.union(occ[0]["initial"], occ[1]["initial"])
        uf.union(occ[0]["terminal"], occ[1]["terminal"])
    return uf.classes(), occurrences


def surface_invariants(name, word):
    classes, occurrences = vertex_classes_for_word(word)
    edge_pair_types = {}
    helper_schema = []
    for label, occ in sorted(occurrences.items()):
        signs = sorted(item["sign"] for item in occ)
        edge_pair_types[label] = "complementary" if signs == [-1, 1] else "twisted"
        for item in occ:
            helper_schema.append(label if item["sign"] > 0 else f"-{label}")
    vertices = len(classes)
    edges = len(occurrences)
    faces = 1
    chi = euler_characteristic(vertices, edges, faces)
    orientable = all(kind == "complementary" for kind in edge_pair_types.values())
    helper_orientable = orientability_from_schema(helper_schema)
    if orientable != helper_orientable:
        raise AssertionError(f"Orientability helper mismatch for {name}")
    if orientable:
        standard_family = "sphere" if chi == 2 else f"orientable genus {(2 - chi) // 2}"
    else:
        standard_family = f"nonorientable genus {2 - chi}"
    return {
        "name": name,
        "word": word_text(word),
        "vertices": vertices,
        "edges": edges,
        "faces": faces,
        "euler_characteristic": chi,
        "orientable": orientable,
        "edge_pair_types": edge_pair_types,
        "vertex_classes": [list(item) for item in classes],
        "standard_family_signal": standard_family,
    }


example_words = {
    "sphere": ["a", "a^-1"],
    "torus": ["a", "b", "a^-1", "b^-1"],
    "projective plane": ["a", "a"],
    "Klein bottle": ["a", "b", "a", "b^-1"],
}

example_invariants = [surface_invariants(name, word) for name, word in example_words.items()]
invariant_path = save_json(example_invariants, CHECKS / "surface-presentation-invariants.json")
example_table_path = save_csv(
    [
        {
            "surface": item["name"],
            "word": item["word"],
            "V": item["vertices"],
            "E": item["edges"],
            "F": item["faces"],
            "Euler characteristic": item["euler_characteristic"],
            "orientable": item["orientable"],
            "classification signal": item["standard_family_signal"],
        }
        for item in example_invariants
    ],
    TABLES / "basic-surface-invariants.csv",
)

display(pd.read_csv(example_table_path))
display_artifact(invariant_path)


In [ ]:
def polygon_vertices(n):
    angles = np.pi / 2 + 2 * np.pi * np.arange(n) / n
    return np.column_stack([np.cos(angles), np.sin(angles)])


def plot_schema(ax, word, title):
    n = len(word)
    vertices = polygon_vertices(n)
    labels = sorted({parse_token(token)[0] for token in word})
    cmap = plt.get_cmap("Set2")
    colors = {label: cmap(i % cmap.N) for i, label in enumerate(labels)}
    closed = np.vstack([vertices, vertices[0]])
    ax.fill(closed[:, 0], closed[:, 1], color="#f7f7f7", zorder=0)
    ax.plot(closed[:, 0], closed[:, 1], color="#202020", lw=1.2, zorder=1)
    ax.scatter(vertices[:, 0], vertices[:, 1], s=18, color="#202020", zorder=3)
    for i, token in enumerate(word):
        label, sign = parse_token(token)
        p0 = vertices[i]
        p1 = vertices[(i + 1) % n]
        start, end = (p0, p1) if sign > 0 else (p1, p0)
        midpoint = 0.5 * (p0 + p1)
        outward = midpoint / (np.linalg.norm(midpoint) + 1e-9)
        text_point = midpoint + 0.18 * outward
        ax.annotate(
            "",
            xy=end,
            xytext=start,
            arrowprops={"arrowstyle": "-|>", "lw": 2.0, "color": colors[label], "shrinkA": 8, "shrinkB": 8},
            zorder=4,
        )
        ax.text(
            text_point[0],
            text_point[1],
            render_token(token),
            ha="center",
            va="center",
            color=colors[label],
            weight="bold",
            fontsize=10,
        )
    inv = surface_invariants(title, word)
    subtitle = f"V={inv['vertices']}, E={inv['edges']}, F=1, chi={inv['euler_characteristic']}; " \
               f"{'orientable' if inv['orientable'] else 'nonorientable'}"
    ax.set_title(f"{title}\n{subtitle}")
    ax.set_aspect("equal")
    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.35, 1.45)
    ax.axis("off")


fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, word) in zip(axes.flat, example_words.items()):
    plot_schema(ax, word, name)
fig.suptitle("Compact surface schemas as edge-paired polygons", y=0.98, fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])
polygon_schema_path = save_matplotlib(fig, FIGURES / "polygon-schema-models.png")
plt.close(fig)

schema_stats = image_stats(polygon_schema_path)
save_json(schema_stats, CHECKS / "polygon-schema-models-image-stats.json")
display_artifact(polygon_schema_path, width=820)


## Connected Sums And Standard Presentations

For one-face presentations, connected sum has a simple symbolic footprint: after choosing disjoint alphabets, concatenate the two boundary words. Geometrically, this corresponds to removing a disk from each surface and gluing the new boundary circles. The Euler characteristic loses two because the two removed disks each remove one 2-cell contribution and the boundary gluing does not add a new face. Thus `Euler(M # N) = Euler(M) + Euler(N) - 2` for compact connected surfaces in this setting.

The standard orientable family is generated by commutator blocks `a_i b_i a_i^-1 b_i^-1`. Each block contributes a handle. The standard nonorientable family is generated by repeated twisted pairs `c_i c_i`. Each block contributes a crosscap. The source chapter uses elementary transformations to explain why mixed sums do not need a separate family: the Klein bottle becomes the connected sum of two projective planes, and a torus summed with a projective plane can be transformed into three projective planes. The plot below is a compact atlas of the standard list, not a drawing of the embedded surfaces. Hover over a point to read its presentation word and invariant data.


In [ ]:
def torus_block(i):
    return [f"a{i}", f"b{i}", f"a{i}^-1", f"b{i}^-1"]


def projective_block(i):
    return [f"c{i}", f"c{i}"]


def standard_orientable_word(genus):
    if genus == 0:
        return ["s", "s^-1"]
    word = []
    for i in range(1, genus + 1):
        word.extend(torus_block(i))
    return word


def standard_nonorientable_word(crosscaps):
    word = []
    for i in range(1, crosscaps + 1):
        word.extend(projective_block(i))
    return word


standard_rows = []
for genus in range(0, 5):
    word = standard_orientable_word(genus)
    inv = surface_invariants("sphere" if genus == 0 else f"orientable genus {genus}", word)
    standard_rows.append({
        "family": "orientable",
        "parameter": genus,
        "surface": "sphere" if genus == 0 else f"#{genus} tori",
        "word": inv["word"],
        "euler_characteristic": inv["euler_characteristic"],
        "orientable": inv["orientable"],
    })
for crosscaps in range(1, 7):
    word = standard_nonorientable_word(crosscaps)
    inv = surface_invariants(f"nonorientable genus {crosscaps}", word)
    standard_rows.append({
        "family": "nonorientable",
        "parameter": crosscaps,
        "surface": f"#{crosscaps} projective planes",
        "word": inv["word"],
        "euler_characteristic": inv["euler_characteristic"],
        "orientable": inv["orientable"],
    })

standard_table_path = save_csv(standard_rows, TABLES / "standard-surface-family-atlas.csv")
standard_df = pd.DataFrame(standard_rows)

atlas = px.scatter(
    standard_df,
    x="parameter",
    y="euler_characteristic",
    color="family",
    symbol="family",
    hover_data={"surface": True, "word": True, "orientable": True, "parameter": True, "family": True},
    labels={"parameter": "genus or crosscap count", "euler_characteristic": "Euler characteristic"},
    title="Standard compact connected surfaces by Euler characteristic and orientability",
)
atlas.update_traces(marker={"size": 13, "line": {"width": 1, "color": "#222"}})
atlas.update_layout(template="plotly_white", height=520)
atlas_path = save_plotly_html(atlas, HTML / "euler-orientability-classification-atlas.html")

g, k = sp.symbols("g k", integer=True, positive=True)
symbolic_checks = {
    "orientable_formula": str(2 - 2 * g),
    "nonorientable_formula": str(2 - k),
    "connected_sum_torus_increment": int((2 - 2 * (3 + 1)) - (2 - 2 * 3)),
    "connected_sum_projective_increment": int((2 - (4 + 1)) - (2 - 4)),
    "torus_and_projective_equal_three_projective_chi": bool(sp.Eq((0 + 1 - 2), 2 - 3)),
}
assert symbolic_checks["connected_sum_torus_increment"] == -2
assert symbolic_checks["connected_sum_projective_increment"] == -1
assert symbolic_checks["torus_and_projective_equal_three_projective_chi"] is True
connected_sum_checks_path = save_json(symbolic_checks, CHECKS / "connected-sum-formula-checks.json")

display(standard_df)
display_artifact(atlas_path, width=860, height=560)
display_artifact(connected_sum_checks_path)


## Proof And Invariant Scaffolds

The classification theorem is not a single visual move. It is a controlled reduction algorithm. The proof begins by using triangulation to obtain a surface presentation. Connectedness lets the presentation be pasted into one face. Folding removes adjacent complementary pairs unless the surface has already reduced to the sphere. Twisted pairs are brought together. Vertex classes are consolidated to one class. Intertwined complementary pairs are isolated as torus blocks. Finally, the lemmas about the Klein bottle and the sum of a torus with a projective plane remove the need for a mixed family.

The dependency graph records this logic as a proof map. The ledger that follows records the Euler-characteristic invariant used later in the chapter. It does not prove homeomorphism invariance of Euler characteristic for all finite CW complexes; the book postpones that proof to homology. Here the ledger has the smaller role used in Chapter 6: each elementary transformation changes cell counts in a way that keeps `V - E + F` unchanged.


In [ ]:
proof_edges = [
    ("compact surface", "triangulation gives finite 2-complex"),
    ("triangulation gives finite 2-complex", "polygonal surface presentation"),
    ("polygonal surface presentation", "one-face presentation"),
    ("one-face presentation", "remove adjacent complementary pairs"),
    ("remove adjacent complementary pairs", "sphere endpoint"),
    ("remove adjacent complementary pairs", "twisted pairs adjacent"),
    ("twisted pairs adjacent", "single vertex class"),
    ("single vertex class", "intertwined complementary pairs exist"),
    ("intertwined complementary pairs exist", "torus blocks isolated"),
    ("torus blocks isolated", "standard torus and projective blocks"),
    ("standard torus and projective blocks", "mixed sums eliminated"),
    ("Klein bottle = P2 # P2", "mixed sums eliminated"),
    ("T2 # P2 = P2 # P2 # P2", "mixed sums eliminated"),
    ("mixed sums eliminated", "standard classification list"),
]
G = nx.DiGraph()
G.add_edges_from(proof_edges)
assert nx.is_directed_acyclic_graph(G)
assert "standard classification list" in G.nodes

try:
    pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
except Exception:
    pos = nx.spring_layout(G, seed=6)

fig, ax = plt.subplots(figsize=(12, 7))
node_colors = []
for node in G.nodes:
    if "endpoint" in node or "standard classification" in node:
        node_colors.append("#f0b35a")
    elif "P2" in node or "T2" in node or "Klein" in node:
        node_colors.append("#b7d8c2")
    else:
        node_colors.append("#c7d7ef")
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14, width=1.1, edge_color="#555")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2400, node_color=node_colors, edgecolors="#222", linewidths=0.8)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
ax.set_title("Proof dependency map for compact surface classification, Part I")
ax.axis("off")
fig.tight_layout()
proof_graph_path = save_matplotlib(fig, FIGURES / "classification-proof-dependency-map.png")
plt.close(fig)
proof_graph_check_path = save_json(
    {
        "node_count": G.number_of_nodes(),
        "edge_count": G.number_of_edges(),
        "is_dag": nx.is_directed_acyclic_graph(G),
        "source_nodes": [node for node, degree in G.in_degree() if degree == 0],
        "sink_nodes": [node for node, degree in G.out_degree() if degree == 0],
    },
    CHECKS / "classification-proof-dependency-map.json",
)

display_artifact(proof_graph_path, width=900)
display_artifact(proof_graph_check_path)


In [ ]:
ledger_rows = [
    {"operation": "relabel", "delta_vertices": 0, "delta_edges": 0, "delta_faces": 0},
    {"operation": "rotate", "delta_vertices": 0, "delta_edges": 0, "delta_faces": 0},
    {"operation": "reflect", "delta_vertices": 0, "delta_edges": 0, "delta_faces": 0},
    {"operation": "subdivide", "delta_vertices": 1, "delta_edges": 1, "delta_faces": 0},
    {"operation": "consolidate", "delta_vertices": -1, "delta_edges": -1, "delta_faces": 0},
    {"operation": "cut", "delta_vertices": 0, "delta_edges": 1, "delta_faces": 1},
    {"operation": "paste", "delta_vertices": 0, "delta_edges": -1, "delta_faces": -1},
    {"operation": "unfold", "delta_vertices": 1, "delta_edges": 1, "delta_faces": 0},
    {"operation": "fold", "delta_vertices": -1, "delta_edges": -1, "delta_faces": 0},
]
for row in ledger_rows:
    row["delta_euler"] = row["delta_vertices"] - row["delta_edges"] + row["delta_faces"]
    assert row["delta_euler"] == 0
ledger_path = save_csv(ledger_rows, TABLES / "elementary-transformation-euler-ledger.csv")
ledger_check_path = save_json(
    {
        "operations_checked": len(ledger_rows),
        "all_delta_euler_zero": all(row["delta_euler"] == 0 for row in ledger_rows),
        "ledger_path": relative(ledger_path),
    },
    CHECKS / "elementary-transformation-euler-ledger.json",
)

display(pd.DataFrame(ledger_rows))
display_artifact(ledger_check_path)


## Orientability From Edge Pairs

The orientation test in this chapter is intentionally combinatorial. A presentation is oriented when no edge pair is twisted. In the word model, that means every label appears once with positive direction and once with inverse direction. The torus word has two complementary pairs, so the top side of the polygon can be colored consistently before gluing. The projective-plane word has one twisted pair, so any attempted front/back coloring reverses when the edge is crossed. The Klein bottle combines one twisted pair and one complementary pair; its Euler characteristic matches the torus, but the orientation flag separates their standard-family signals.

This is a good example of why one invariant is not enough. Euler characteristic zero points to either the torus or the Klein bottle. Orientability supplies the missing presentation-level information: orientable with Euler characteristic zero gives genus one in the orientable family; nonorientable with Euler characteristic zero gives nonorientable genus two. Later chapters will prove that orientability is genuinely topological, not merely a feature of the chosen word.


## Applied Lab: Classifying Presentation Words

The lab below tests a small set of one-face surface presentations. Each word is deliberately short enough to inspect by hand, but the computations mirror the classification workflow: build quotient vertex classes, compute `V - E + F`, check edge-pair orientation, and return the standard-family signal. The words are not copied from the problem set. They are chosen to exercise the same mechanisms: a sphere-like cancellation, an orientable handle block, a crosscap block, and mixed or repeated blocks.

Try editing the `lab_words` dictionary and rerunning the cell. A valid surface presentation must use every edge label exactly twice. If a label occurs once or three times, the code refuses to classify it because it is no longer modeling a compact surface obtained by paired boundary edges.


In [ ]:
lab_words = {
    "two handles": ["a", "b", "a^-1", "b^-1", "c", "d", "c^-1", "d^-1"],
    "three crosscaps": ["a", "a", "b", "b", "c", "c"],
    "torus plus projective signal": ["a", "b", "a^-1", "b^-1", "c", "c"],
    "Klein bottle schema": ["a", "b", "a", "b^-1"],
    "sphere with redundant pair model": ["a", "b", "b^-1", "a^-1"],
}

lab_results = [surface_invariants(name, word) for name, word in lab_words.items()]
lab_rows = []
for item in lab_results:
    lab_rows.append({
        "example": item["name"],
        "word": item["word"],
        "vertex_classes": item["vertex_classes"],
        "V": item["vertices"],
        "E": item["edges"],
        "F": item["faces"],
        "Euler characteristic": item["euler_characteristic"],
        "orientable": item["orientable"],
        "standard-family signal": item["standard_family_signal"],
    })
lab_table_path = save_csv(lab_rows, TABLES / "applied-lab-classification-results.csv")
lab_check_path = save_json(lab_results, CHECKS / "applied-lab-classification-results.json")

lab_df = pd.DataFrame(lab_rows)
display(lab_df)
display_artifact(lab_check_path)

assert lab_df.loc[lab_df["example"] == "two handles", "Euler characteristic"].iloc[0] == -2
assert lab_df.loc[lab_df["example"] == "three crosscaps", "Euler characteristic"].iloc[0] == -1
assert lab_df.loc[lab_df["example"] == "Klein bottle schema", "orientable"].iloc[0] == False


## Final Sanity Checks

The final cell checks the notebook's contract rather than the whole theory of compact surfaces. It asserts that core examples have the expected presentation invariants, that the symbolic formulas match the standard families, that all planned artifacts exist and are nonempty, and that generated PNG files are not blank. It also writes a final JSON summary under the chapter checks directory.


In [ ]:
expected = {
    "sphere": {"euler_characteristic": 2, "orientable": True},
    "torus": {"euler_characteristic": 0, "orientable": True},
    "projective plane": {"euler_characteristic": 1, "orientable": False},
    "Klein bottle": {"euler_characteristic": 0, "orientable": False},
}
by_name = {item["name"]: item for item in example_invariants}
for name, values in expected.items():
    for key, expected_value in values.items():
        assert by_name[name][key] == expected_value, (name, key, by_name[name][key], expected_value)

for row in standard_rows:
    if row["family"] == "orientable":
        assert row["euler_characteristic"] == 2 - 2 * row["parameter"]
        assert row["orientable"] is True
    else:
        assert row["euler_characteristic"] == 2 - row["parameter"]
        assert row["orientable"] is False

planned_artifacts = [
    visual_storyboard_path,
    library_routing_path,
    invariant_path,
    example_table_path,
    polygon_schema_path,
    atlas_path,
    connected_sum_checks_path,
    standard_table_path,
    proof_graph_path,
    proof_graph_check_path,
    ledger_path,
    ledger_check_path,
    lab_table_path,
    lab_check_path,
]
assert_artifacts(planned_artifacts, min_bytes=64)

png_stats = [image_stats(polygon_schema_path), image_stats(proof_graph_path)]
for stats in png_stats:
    assert stats["bytes"] > 1000
    assert stats["width"] >= 400 and stats["height"] >= 300
    assert stats["max_channel_stddev"] > 2.0

final_sanity = {
    "chapter": "Chapter 6: Compact Surfaces",
    "source_span": "printed pages 159-182; PDF pages 177-200",
    "artifact_count_checked": len(planned_artifacts),
    "png_stats": png_stats,
    "core_examples_checked": expected,
    "standard_family_rows": len(standard_rows),
    "elementary_transformations_delta_euler_zero": all(row["delta_euler"] == 0 for row in ledger_rows),
    "classification_graph_is_dag": nx.is_directed_acyclic_graph(G),
    "known_theory_boundary": "Part I classification is represented; later chapters prove distinctness of the standard surfaces.",
}
final_sanity_path = save_json(final_sanity, CHECKS / "final-sanity.json")
assert_artifacts([final_sanity_path], min_bytes=64)
display_artifact(final_sanity_path)
final_sanity


## Takeaways

A compact surface can be encoded by finite gluing data. The polygon schema is not just a picture; it is a quotient recipe whose vertices, edges, and faces can be counted after the identifications are imposed.

Connected sum becomes word concatenation for one-face presentations. This makes the standard surfaces easy to list: the sphere, repeated commutator blocks for orientable genus, and repeated twisted pairs for nonorientable genus.

Euler characteristic is the first fast invariant in the chapter. Elementary transformations preserve it because their changes in vertex, edge, and face counts cancel. For standard presentations, the formulas are `2`, `2 - 2g`, and `2 - k`.

Orientability is the second fast signal. Complementary edge pairs support a consistent orientation of polygon faces; twisted edge pairs obstruct it. Together with Euler characteristic, this separates the torus signal from the Klein bottle signal at the presentation level.

The classification theorem in this chapter is a reduction theorem: every compact connected surface reaches the standard list. The proof that the standard surfaces are genuinely distinct is intentionally deferred to the later algebraic-topology tools.
